In [10]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


---

In [11]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [12]:
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

In [13]:
from ldpc.bp_decoder import BpDecoder

---

In [14]:
import numpy as np
from utils.LDPC_encode import QCLDPCEncoder

base_pc_path = '../pc_matrices/matlab_h.txt'

P = np.loadtxt(base_pc_path, dtype=int)
blocksize = 27

encoder = QCLDPCEncoder(base_matrix= P, Z= blocksize)

Initializing Encoder: Full Matrix Size 162x648, Message Bits: 486
  > Inverting Parity Matrix (this may take a moment for large Z)...
  > Computing Generator Matrix...
Encoder Ready.


In [15]:
decoder = BpDecoder(encoder.H, schedule="cluster")
max_iter = 5

In [ ]:
k = (P.shape[1] - P.shape[0]) * blocksize
n_frames = 10000


message = np.random.randint(0, 2, (n_frames, k))
print("Message shape:", message.shape)

codeword = encoder.encode(message)
tx_codeword = 1 - 2 * codeword

Message shape: (1000, 486)


In [17]:
m = P.shape[0] * blocksize
check_nodes = np.arange(m)
clusters = check_nodes.reshape(P.shape[0], -1)

In [18]:
snrs = [3, 4, 5, 6, 7]
bers = []

for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        for iter in range(max_iter):
            for cluster in clusters:
                llr = decoder.decode_cluster(cluster)
        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :k]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR 3 dB: 0.07108847736625515
BER at SNR 4 dB: 0.026088477366255143
BER at SNR 5 dB: 0.0008456790123456791
BER at SNR 6 dB: 0.0
BER at SNR 7 dB: 0.0
